# Module 05 — Lesson 03
# Reading CSV and Excel Files

---

> In Lesson 02 you built every DataFrame by hand -- typing out dicts and lists yourself. Real datasets don't arrive that way. They arrive as a `.csv` you downloaded, or an `.xlsx` a client emailed you. This lesson is where that file on your hard drive becomes a DataFrame you can actually work with -- the single most-repeated first step in every project in this course.

We'll use a small real dataset: `tips.csv`, a restaurant's log of bills and tips (244 rows) -- the same one you'll see reused throughout the module.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Load a real CSV file into a DataFrame with `pd.read_csv()`, and use its key parameters: `sep`, `header`, `index_col`, `usecols`, `nrows`
- Load an Excel workbook with `pd.read_excel()`, including picking a specific sheet with `sheet_name`
- Save a DataFrame back to disk with `.to_csv()` and `.to_excel()` without creating a stray index column
- Apply the "first look" habit (`.head()`, `.info()`, `.shape`, `.describe()`) to a freshly loaded real dataset
- Diagnose and fix the most common file-loading errors: wrong path, wrong separator, wrong sheet

---
## 1. `pd.read_csv()` — The Basics

`pd.read_csv()` takes a file path (or a URL) and returns a DataFrame. That's it for the simple case -- one line of code replaces writing your own file parser.

This notebook assumes it's run from its own folder, so the path `data/tips.csv` is relative to `lesson-03_read-csv-excel/`.

In [ ]:
import pandas as pd

tips = pd.read_csv("data/tips.csv")
print(tips.head())
print()
print(f"type: {type(tips)}")

---
## 2. Useful `read_csv()` Parameters

Real files are rarely "just read the whole thing." These five parameters cover almost every situation you'll hit:

| Parameter | What it does |
|---|---|
| `sep` | The delimiter between values (default `,`) -- use `;`, `\t`, etc. for other formats |
| `header` | Which row holds the column names (default: row 0) |
| `index_col` | Which column to use as the row index instead of 0, 1, 2... |
| `usecols` | Load only specific columns -- faster and less memory for wide files |
| `nrows` | Load only the first N rows -- great for peeking at a huge file before committing to load it all |

In [ ]:
# usecols + nrows: peek at just 3 columns, first 5 rows -- handy for a huge file
preview = pd.read_csv("data/tips.csv", usecols=["total_bill", "tip", "day"], nrows=5)
print("usecols + nrows:")
print(preview)

In [ ]:
# index_col: use an existing column as the row label instead of 0, 1, 2...
tips_by_bill = pd.read_csv("data/tips.csv", index_col="total_bill")
print("index_col='total_bill':")
print(tips_by_bill.head(3))

In [ ]:
# sep: a file doesn't have to be comma-separated. Here's a tiny semicolon-separated
# file simulated in memory with io.StringIO, just to show sep in action.
import io

semicolon_data = """name;age;city
Aisha;20;Pune
Ben;21;Delhi
"""

custom_sep_df = pd.read_csv(io.StringIO(semicolon_data), sep=";")
print("sep=';':")
print(custom_sep_df)

---
## 3. `pd.read_excel()` — Reading Excel Files

Excel files can hold **multiple sheets** in one workbook, so `pd.read_excel()` adds a `sheet_name` parameter on top of everything `read_csv()` already does. By default it reads the *first* sheet.

Reading `.xlsx` files requires the `openpyxl` package installed (`pip install openpyxl`) -- pandas uses it under the hood.

In [ ]:
# By default, read_excel() loads the FIRST sheet in the workbook
tips_first_sheet = pd.read_excel("data/tips.xlsx")
print(f"Default sheet shape: {tips_first_sheet.shape}")
print()

# Our workbook actually has two sheets: 'tips' (all 244 rows) and 'tips_sample' (first 20)
# Pass sheet_name explicitly to pick the one you want
tips_sample = pd.read_excel("data/tips.xlsx", sheet_name="tips_sample")
print(f"'tips_sample' sheet shape: {tips_sample.shape}")
print(tips_sample.head(3))

In [ ]:
# Not sure what sheets exist? Ask the file itself with pd.ExcelFile
workbook = pd.ExcelFile("data/tips.xlsx")
print(f"Sheets in tips.xlsx: {workbook.sheet_names}")

---
## 4. Saving DataFrames Back to Disk

Once you've cleaned or filtered a DataFrame, `.to_csv()` and `.to_excel()` write it back out. The one detail to get right every time: **`index=False`** -- otherwise pandas writes its own row-number index as an extra column in the file.

In [ ]:
dinner_only = tips[tips["time"] == "Dinner"]
print(f"Dinner rows: {dinner_only.shape[0]} out of {tips.shape[0]} total")

dinner_only.to_csv("data/dinner_only.csv", index=False)
dinner_only.to_excel("data/dinner_only.xlsx", index=False, sheet_name="dinner")
print("Saved data/dinner_only.csv and data/dinner_only.xlsx")

---
## 5. The First-Look Habit, Applied to Real Data

Every time you load an external file, run these four checks immediately -- from Lesson 02, but now on data you didn't create yourself, so nothing about it is guaranteed.

In [ ]:
print(f"tips.shape: {tips.shape}")
print()
print("tips.dtypes:")
print(tips.dtypes)
print()
tips.info()
print()
print("tips.describe():")
print(tips.describe())

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Forgetting index=False when saving -----------------------------

print("Mistake 1 — saving WITHOUT index=False bakes the row index into the file:")
tips.head(3).to_csv("data/_bad_example.csv")           # no index=False
reloaded_bad = pd.read_csv("data/_bad_example.csv")
print("Reloading it back shows a stray 'Unnamed: 0' column:")
print(reloaded_bad)
print()
print("Correct: always pass index=False unless you deliberately want the index saved.")

import os
os.remove("data/_bad_example.csv")  # clean up the demo file

In [ ]:
# -- Mistake 2: Wrong file path ------------------------------------------------

print("Mistake 2 — a path that's relative to the wrong folder raises FileNotFoundError:")
try:
    pd.read_csv("tips.csv")   # missing the 'data/' folder in the path
except FileNotFoundError as e:
    print(f"  Caught error: {e}")
print()
print("  Correct: check your working directory with os.getcwd(), and use a path")
print("  relative to it (or an absolute path) -- here that's 'data/tips.csv'.")
print(f"  Current working directory: {os.getcwd()}")

In [ ]:
# -- Mistake 3: Assuming read_excel() gives you the sheet you meant -------------

print("Mistake 3 — read_excel() silently loads the FIRST sheet if you don't ask:")
default_sheet = pd.read_excel("data/tips.xlsx")
print(f"  No sheet_name given -> shape {default_sheet.shape} (this is the 'tips' sheet)")
print("  If you actually wanted 'tips_sample', you'd get the wrong 244-row sheet")
print("  instead of the 20-row one, with no error to warn you.")
print()
print("  Correct: always pass sheet_name explicitly when a workbook has more than one sheet.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Load and Inspect

Read `data/tips.csv` into a DataFrame called `tips_df`. Print its `.shape` and the first 5 rows with `.head()`.

In [ ]:
# Your code here

### Exercise 2 — Selective Loading

Using `pd.read_csv()`, load only the `"total_bill"`, `"tip"`, and `"size"` columns from `data/tips.csv`, and only the first 10 rows. Print the result.

In [ ]:
# Your code here

### Exercise 3 — Reading a Specific Excel Sheet

Read the `"tips_sample"` sheet from `data/tips.xlsx` into a DataFrame. Print its `.shape` and confirm it has fewer rows than the full `tips.csv`.

In [ ]:
# Your code here

### Exercise 4 — Filter and Save

From `tips_df` (Exercise 1), keep only rows where `"smoker"` is `"Yes"`. Save the result to `data/smokers_only.csv` with `index=False`. Then reload that file and print its `.shape` to confirm it matches.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Every Sheet at Once

Pass `sheet_name=None` to `pd.read_excel()` on `data/tips.xlsx` -- this returns a **dict** of `{sheet_name: DataFrame}` for every sheet in the workbook. Loop over the dict and print each sheet's name and its `.shape`.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`pd.read_csv()` and `pd.read_excel()` are how virtually every real dataset enters your notebook** — knowing `sep`, `header`, `index_col`, `usecols`, `nrows`, and `sheet_name` covers nearly every file you'll be handed in this course.

- **Always pass `index=False` when saving with `.to_csv()` / `.to_excel()`**, unless you deliberately want the row index written — otherwise every reload picks up a stray `"Unnamed: 0"` column.

- **Run `.head()`, `.info()`, and `.shape` the moment you load an external file** — unlike the DataFrames you built by hand in Lesson 02, a real file can have unexpected dtypes, missing values, or the wrong sheet loaded, and these checks catch it immediately.

---
## What's Next?

**Lesson 04 — Selecting Rows and Columns, Filtering** takes the `tips` DataFrame you can now load from disk and teaches you how to ask it real questions -- "show me only the dinner bills," "show me only parties of 4 or more."